# 03 - Triple Barrier Labeling

Implementação do método de labeling baseado em:
- **Take Profit**: Barreira superior
- **Stop Loss**: Barreira inferior
- **Timeout**: Barreira vertical (tempo)

**WIN**: atingiu TP antes de SL ou timeout  
**LOSS**: atingiu SL ou timeout sem TP

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use('dark_background')
DATA_DIR = Path('../data')

## 1. Implementação Triple Barrier

In [ ]:
def triple_barrier_label(
    entry_price: float,
    action: str,
    exit_price: float,
    stop_loss: float,
    take_profit: float,
    time_to_result: float,
    max_time: float = 45.0
) -> int:
    """
    Aplica Triple Barrier Method.
    
    Args:
        entry_price: Preço de entrada
        action: 'BUY' ou 'SELL'
        exit_price: Preço de saída
        stop_loss: Preço do stop loss
        take_profit: Preço do take profit
        time_to_result: Tempo até resultado (minutos)
        max_time: Timeout máximo (minutos)
    
    Returns:
        1 = WIN (atingiu TP)
        0 = LOSS (atingiu SL ou timeout)
    """
    # Timeout = LOSS
    if time_to_result >= max_time:
        return 0
    
    if action == 'BUY':
        # LONG: WIN se preço subiu até TP
        if exit_price >= take_profit:
            return 1
        elif exit_price <= stop_loss:
            return 0
    else:  # SELL
        # SHORT: WIN se preço caiu até TP
        if exit_price <= take_profit:
            return 1
        elif exit_price >= stop_loss:
            return 0
    
    # Se não atingiu nenhuma barreira e não deu timeout
    # Verificar profit/loss baseado no movimento
    if action == 'BUY':
        profit = (exit_price - entry_price) / entry_price
    else:
        profit = (entry_price - exit_price) / entry_price
    
    return 1 if profit > 0 else 0

## 2. Aplicar nos Dados

In [ ]:
df = pd.read_parquet(DATA_DIR / 'signals.parquet')
print(f'Total: {len(df)} sinais')

# Aplicar Triple Barrier
df['label_triple'] = df.apply(
    lambda row: triple_barrier_label(
        entry_price=row['entry_price'],
        action=row['action'],
        exit_price=row['exit_price'],
        stop_loss=row['stop_loss'],
        take_profit=row['take_profit'],
        time_to_result=row['time_to_result_min'],
        max_time=45.0
    ),
    axis=1
)

print(f'\n=== Comparação Labels ===')
print(f'Label Original (is_win): {df["is_win"].sum()} WIN')
print(f'Label Triple Barrier: {df["label_triple"].sum()} WIN')

In [ ]:
# Comparar distribuições
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Original
df['is_win'].value_counts().plot(
    kind='pie', 
    ax=axes[0], 
    autopct='%1.1f%%',
    labels=['LOSS', 'WIN'],
    colors=['red', 'lime']
)
axes[0].set_title('Label Original')

# Triple Barrier
df['label_triple'].value_counts().plot(
    kind='pie', 
    ax=axes[1], 
    autopct='%1.1f%%',
    labels=['LOSS', 'WIN'],
    colors=['red', 'lime']
)
axes[1].set_title('Label Triple Barrier')

plt.tight_layout()
plt.show()

## 3. Análise de Timeout

In [ ]:
# Quantos sinais deram timeout?
timeout_mask = df['time_to_result_min'] >= 45
print(f'Sinais com Timeout (>= 45 min): {timeout_mask.sum()} ({timeout_mask.mean()*100:.1f}%)')
print(f'Win Rate nos Timeouts: {df[timeout_mask]["is_win"].mean()*100:.1f}%')
print(f'Win Rate sem Timeouts: {df[~timeout_mask]["is_win"].mean()*100:.1f}%')

In [ ]:
# Distribuição do tempo até resultado
fig, ax = plt.subplots(figsize=(12, 5))
df['time_to_result_min'].hist(bins=50, ax=ax, color='cyan', alpha=0.7)
ax.axvline(45, color='red', linestyle='--', label='Timeout (45 min)')
ax.set_xlabel('Tempo até resultado (min)')
ax.set_ylabel('Frequência')
ax.set_title('Distribuição do Tempo até Resultado')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Salvar Labels

In [ ]:
# Salvar dataset com novo label
df['y'] = df['label_triple']  # Target final

output_path = DATA_DIR / 'signals_labeled.parquet'
df.to_parquet(output_path, index=False)
print(f'Dataset salvo em: {output_path}')
print(f'Target (y): {df["y"].sum()} WIN / {(~df["y"].astype(bool)).sum()} LOSS')

## 5. Próximos Passos

1. **04_feature_eng.ipynb**: Separar X_regime e X_signal
2. **05_train_gmm.ipynb**: Treinar GMM para regime detection